# Simple RAG — Groq + Pinecone + Ollama

Run cells **in order** from top to bottom.

In [6]:
# 1) Load environment variables from .env
import os
from pathlib import Path

ENV_FILE = Path("/agent369-Raja/.env")

for raw in ENV_FILE.read_text().splitlines():
    line = raw.strip()
    if not line or line.startswith("#"):
        continue
    if line.startswith("export "):
        line = line.removeprefix("export ").strip()
    key, sep, val = line.partition("=")
    if not sep:
        continue
    os.environ.setdefault(key.strip(), val.strip().strip('"').strip("'"))

# Fix LangSmith 405: disable tracing (web UI URL is not the API)
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ.setdefault("LANGCHAIN_PROJECT", "rag-csv-project")

required_keys = ["GROQ_API_KEY", "PINECONE_API_KEY", "LANGCHAIN_API_KEY"]
missing = [k for k in required_keys if not os.getenv(k)]
if missing:
    raise ValueError(f"Missing keys in .env: {', '.join(missing)}")

print("Environment OK")

Environment OK


In [7]:
# 2) Install dependencies (run once)
!pip install -q \
  langchain==0.1.17 \
  langchain-core==0.1.52 \
  langchain-community==0.0.38 \
  langchain-pinecone==0.1.2 \
  langchain-groq \
  pydantic==1.10.13

## Phase 1 — Ingest documents into Pinecone

In [8]:
# 3) Ingestion: load text file, chunk, embed, upsert to Pinecone
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

DATA_FILE_PATH = "/agent369-Raja/data/data.csv"  # markdown/text content
INDEX_NAME = "csv-rag-index"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
BATCH_SIZE = 100

embeddings = OllamaEmbeddings(model="nomic-embed-text")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

docs = TextLoader(DATA_FILE_PATH, encoding="utf-8").load()
for d in docs:
    d.metadata.update({"source": "acrisma-dataset", "type": "text"})

chunks = splitter.split_documents(docs)
print(f"Loaded {len(docs)} doc(s), split into {len(chunks)} chunks")

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

vectorstore = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings)

for i in range(0, len(chunks), BATCH_SIZE):
    vectorstore.add_documents(chunks[i : i + BATCH_SIZE])

print(f"Indexed {len(chunks)} chunks into Pinecone index '{INDEX_NAME}'")

Loaded 1 doc(s), split into 11 chunks


Indexed 11 chunks into Pinecone index 'csv-rag-index'


## Phase 2 — Retrieval + generation

In [10]:
# 4) Build RAG chain (simple retriever — avoids EmbeddingsFilter/simsimd TypeError)
import os

os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def build_rag_input(question: str) -> dict:
    docs = retriever.invoke(question)
    return {"context": format_docs(docs), "question": question}

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

prompt = ChatPromptTemplate.from_template("""
Use the context below to answer the question in 2-4 sentences.
If the context mentions the topic, summarize what it says.
Only reply with exactly "I don't know" if the context has zero relevant information.

Context:
{context}

Question:
{question}
""")

rag_chain = RunnableLambda(build_rag_input) | prompt | llm | StrOutputParser()

def guardrail(text: str) -> str:
    text = text.strip()
    if len(text) < 10:
        return "Low confidence answer"
    # Only flag when the model gives up entirely (not partial "don't know" phrases)
    if text.lower() in ("i don't know", "i don't know.", "i do not know"):
        return "No answer found in context"
    return text

print("RAG chain ready")

RAG chain ready


## Phase 3 — Test

In [15]:
# 5) Ask a question
query = "What is the ACRIMA dataset?"

# Debug: see what was retrieved (run cell 4 first if this is empty)
retrieved = retriever.invoke(query)
print(f"Retrieved {len(retrieved)} chunks, {sum(len(d.page_content) for d in retrieved)} chars total")
if not retrieved:
    print("WARNING: No chunks retrieved. Re-run cell 4 (ingestion) after kernel restart.")
else:
    print("Sample:", retrieved[0].page_content[:200], "...\n")

answer = rag_chain.invoke(query)
print("Raw answer:", answer)
print("Final:", guardrail(answer))

Retrieved 8 chunks, 269 chars total
Sample: # ACRIMA: The ACRIMA dataset contains 705 fundus images ...

Raw answer: The ACRIMA dataset contains 705 fundus images.
Final: The ACRIMA dataset contains 705 fundus images.
